In [0]:
from pyspark.sql import functions as F

T_trans = spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
T_locations = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_location")
T_date = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_date")
T_subscriber = spark.table("cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot")
T_events = spark.table("cpx_dataanalytics_gold.marketing.mkt_subscription_events")

In [0]:
# Clears all cached Spark DataFrames in the session
spark.catalog.clearCache()

In [0]:
Total_Population = (
    T_trans
    .join(T_date.select("dateid", "datefull"), T_trans.VisitDateId == T_date.dateid, "left")
    .join(T_locations.select("LocationID", "LC_Location_Type_Description"), T_trans.LocationId == T_locations.LocationID, "left")
    .filter(F.col("LC_Location_Type_Description") == "Cineplex Theatre")
    .filter(F.col("TicketFLAG")== "Yes")
    .filter(F.col("CineClubFLAG") == "No")
    .withColumn("Year", F.year(F.col("datefull")))
    .groupBy("Year")
    .agg(
        F.sum("Quantity").alias("Total_Tickets"),
        F.sum("Net_Revenue").alias("Net_Revenue_Tickets")
    )
)

In [0]:
Total_Population.display()

In [0]:
Total_Population_Conc = (
    T_trans
    .join(T_date.select("dateid", "datefull"), T_trans.VisitDateId == T_date.dateid, "left")
    .join(T_locations.select("LocationID", "LC_Location_Type_Description"), T_trans.LocationId == T_locations.LocationID, "left")
    .filter(F.col("datefull").between("2019-01-01", "2025-12-31"))
    .filter(F.col("LC_Location_Type_Description") == "Cineplex Theatre")
    .filter(F.col("ConcessionFLAG")== "Yes")
    .filter(F.col("CineClubFLAG") == "No")
    .withColumn("Year", F.year(F.col("datefull")))
    .groupBy("Year")
    .agg(
        F.sum("Net_Revenue").alias("Net_Revenue_Concessions")
    )
)

In [0]:
    Total_Population_Conc.display()

In [0]:
from pyspark.sql import functions as F
from functools import reduce

years = [2018,2019,2020, 2021, 2022, 2023, 2024, 2025]  # edit as needed
outs = []

for Y in years:
    start_id = Y*10000 + 101       # YYYY0101
    end_id   = (Y+1)*10000 + 101   # (YYYY+1)0101 (exclusive)

    base_y = (
        T_trans
        .filter((F.col("VisitDateID") >= start_id) & (F.col("VisitDateID") < end_id))
        .select(
            "cde_id", "CineClubFLAG", "VisitDateID", "LocationId",
        "TicketFLAG", "ConcessionFLAG", "Net_Revenue", "Quantity", "SCENEFLAG"
        )
    )

    # Cohort: CC-only within the year (NULL treated as non-CC, matching your table)
    non_cineclub_ids_y = (
        base_y
        .filter(F.col("SCENEFLAG") == "Yes")
        .filter(F.col("CineClubFLAG") != "Yes")
        .select("cde_id")
        .distinct()
    )

    cc_only_ids_y = (
        base_y
        .filter(F.col("SCENEFLAG") == "Yes")
        .filter(F.col("CineClubFLAG") == "Yes")
        .select("cde_id")
        .distinct()
        .join(non_cineclub_ids_y, "cde_id", "left_anti")
    )

    # Transactions for those members in that year
    tx_y = base_y.join(cc_only_ids_y, "cde_id", "inner")

    # Visits = distinct (cde_id, VisitDateID, LocationId)
    visits_y = tx_y.select("cde_id", "VisitDateID", "LocationId").distinct()

    out_y = (
        tx_y
        .agg(
            F.countDistinct("cde_id").alias("Members"),

            # Visits (distinct by your definition)
            F.countDistinct(F.struct("cde_id", "VisitDateID", "LocationId")).alias("Total_Visits"),
            F.sum(F.when(F.col("TicketFLAG") == "Yes", F.coalesce(F.col("Quantity"), F.lit(0.0))).otherwise(F.lit(0.0))).alias("Total_Tickets"),
            # Net revenues
            F.sum(F.when(F.col("TicketFLAG") == "Yes", F.coalesce(F.col("Net_Revenue"), F.lit(0.0))).otherwise(F.lit(0.0))).alias("Net_Ticket_Revenue"),
            F.sum(F.when(F.col("ConcessionFLAG") == "Yes", F.coalesce(F.col("Net_Revenue"), F.lit(0.0))).otherwise(F.lit(0.0))).alias("Net_Concession_Revenue"),
        )
        .withColumn("Year", F.lit(Y))
        .select("Year", "Members", "Total_Visits", "Total_Tickets", "Net_Ticket_Revenue", "Net_Concession_Revenue")
    )

    outs.append(out_y)

df_yearly = reduce(lambda a, b: a.unionByName(b), outs)


In [0]:
display(df_yearly.orderBy("Year"))

In [0]:
from pyspark.sql import functions as F
from functools import reduce

years = [2018,2019,2020, 2021, 2022, 2023, 2024, 2025]  # edit as needed
outs = []

for Y in years:
    start_id = Y * 10000 + 101       # YYYY0101
    end_id   = (Y + 1) * 10000 + 101 # (YYYY+1)0101 (exclusive)

    base_y = (
        T_trans
        .filter((F.col("VisitDateID") >= start_id) & (F.col("VisitDateID") < end_id))
        .select(
            "cde_id", "CineClubFLAG", "VisitDateID", "LocationId",
            "TicketFLAG", "ConcessionFLAG", "Net_Revenue", "Quantity", "SCENEFLAG"
        )
    )

    # Cohort: Non-CC-only within the year
    # NULL is treated as non-CC here as well
    cineclub_ids_y = (
        base_y
        .filter(F.col("SCENEFLAG") == "Yes")
        .filter(F.col("CineClubFLAG") == "Yes")
        .select("cde_id")
        .distinct()
    )

    non_cc_only_ids_y = (
        base_y
        .filter((F.col("CineClubFLAG") != "Yes") | F.col("CineClubFLAG").isNull())
        .filter(F.col("SCENEFLAG") == "Yes")
        .select("cde_id")
        .distinct()
        .join(cineclub_ids_y, "cde_id", "left_anti")
    )

    # Transactions for those members in that year
    tx_y = base_y.join(non_cc_only_ids_y, "cde_id", "inner")

    out_y = (
        tx_y
        .agg(
            F.countDistinct("cde_id").alias("Members"),

            # Visits = distinct (cde_id, VisitDateID, LocationId)
            F.countDistinct(F.struct("cde_id", "VisitDateID", "LocationId")).alias("Total_Visits"),

            F.sum(F.when(F.col("TicketFLAG") == "Yes", F.coalesce(F.col("Quantity"), F.lit(0.0))).otherwise(F.lit(0.0))).alias("Total_Tickets"),

            # Net revenues
            F.sum(
                F.when(
                    F.col("TicketFLAG") == "Yes",
                    F.coalesce(F.col("Net_Revenue"), F.lit(0.0))
                ).otherwise(F.lit(0.0))
            ).alias("Net_Ticket_Revenue"),

            F.sum(
                F.when(
                    F.col("ConcessionFLAG") == "Yes",
                    F.coalesce(F.col("Net_Revenue"), F.lit(0.0))
                ).otherwise(F.lit(0.0))
            ).alias("Net_Concession_Revenue"),
        )
        .withColumn("Year", F.lit(Y))
        .select("Year", "Members", "Total_Visits", "Total_Tickets", "Net_Ticket_Revenue", "Net_Concession_Revenue")
    )

    outs.append(out_y)

df_yearly_non_cc = reduce(lambda a, b: a.unionByName(b), outs)
display(df_yearly_non_cc.orderBy("Year"))

In [0]:
from pyspark.sql import functions as F
from functools import reduce

base_year = 2025
compare_years = [2018,2019,2020,2021, 2022, 2023, 2024]

start_2025 = base_year * 10000 + 101
end_2025   = (base_year + 1) * 10000 + 101

# ----------------------------
# Base 2025 transactions
# ----------------------------
base_2025 = (
    T_trans
    .filter((F.col("VisitDateID") >= start_2025) & (F.col("VisitDateID") < end_2025))
    .select(
        "cde_id","CineClubFLAG","VisitDateID","LocationId",
        "TicketFLAG","ConcessionFLAG","Net_Revenue", "SCENEFLAG"
    )
)

# ----------------------------
# Pure CineClub members in 2025
# ----------------------------
non_cc_ids_2025 = (
    base_2025
    .filter((F.col("CineClubFLAG") != "Yes") | F.col("CineClubFLAG").isNull())
    .filter(F.col("SCENEFLAG") == "Yes")
    .select("cde_id")
    .distinct()
)

pure_cc_2025 = (
    base_2025
    .filter(F.col("CineClubFLAG") == "Yes")
    .filter(F.col("SCENEFLAG") == "Yes")
    .select("cde_id")
    .distinct()
    .join(non_cc_ids_2025, "cde_id", "left_anti")
)

outs = []

for Y in compare_years:

    start_y = Y * 10000 + 101
    end_y   = (Y + 1) * 10000 + 101

    base_y = (
        T_trans
        .filter((F.col("VisitDateID") >= start_y) & (F.col("VisitDateID") < end_y))
        .select("cde_id","CineClubFLAG", "SCENEFLAG")
    )

    # Members who existed in that year
    existed_ids = base_y.select("cde_id").distinct()

    # Members who were CC in that year
    cc_ids = (
        base_y
        .filter(F.col("CineClubFLAG") == "Yes")
        .filter(F.col("SCENEFLAG") == "Yes")
        .select("cde_id")
        .distinct()
    )

    # Target cohort
    target_ids = (
        pure_cc_2025
        .join(existed_ids, "cde_id", "inner")
        .join(cc_ids, "cde_id", "left_anti")
    )

    # 2025 KPI calculation
    tx_2025_target = base_2025.join(target_ids, "cde_id", "inner")

    out = (
        tx_2025_target
        .agg(
            F.countDistinct("cde_id").alias("Members"),
            F.countDistinct(F.struct("cde_id","VisitDateID","LocationId")).alias("Total_Visits"),
            F.sum(
                F.when(F.col("TicketFLAG")=="Yes",F.coalesce(F.col("Net_Revenue"),F.lit(0.0)))
            ).alias("Net_Ticket_Revenue"),
            F.sum(
                F.when(F.col("ConcessionFLAG")=="Yes",F.coalesce(F.col("Net_Revenue"),F.lit(0.0)))
            ).alias("Net_Concession_Revenue")
        )
        .withColumn("Base_Year",F.lit(2025))
        .withColumn("Compared_Year",F.lit(Y))
    )

    outs.append(out)

df_result = reduce(lambda a,b: a.unionByName(b), outs)

display(df_result.orderBy("Compared_Year"))

In [0]:
from pyspark.sql import functions as F
from functools import reduce

base_year = 2025
compare_years = [2018, 2019, 2020, 2021, 2022, 2023, 2024]

start_2025 = base_year * 10000 + 101
end_2025   = (base_year + 1) * 10000 + 101

# ----------------------------
# Base 2025 transactions
# ----------------------------
base_2025 = (
    T_trans
    .filter((F.col("VisitDateID") >= start_2025) & (F.col("VisitDateID") < end_2025))
    .select(
        "cde_id", "CineClubFLAG", "VisitDateID", "LocationId",
        "TicketFLAG", "ConcessionFLAG", "Net_Revenue", "SCENEFLAG"
    )
)

# ----------------------------
# Pure CineClub members in 2025
# ----------------------------
non_cc_ids_2025 = (
    base_2025
    .filter((F.col("CineClubFLAG") != "Yes") | F.col("CineClubFLAG").isNull())
    .filter(F.col("SCENEFLAG") == "Yes")
    .select("cde_id")
    .distinct()
)

pure_cc_2025 = (
    base_2025
    .filter(F.col("CineClubFLAG") == "Yes")
    .filter(F.col("SCENEFLAG") == "Yes")
    .select("cde_id")
    .distinct()
    .join(non_cc_ids_2025, "cde_id", "left_anti")
)

outs = []

for Y in compare_years:

    start_y = Y * 10000 + 101
    end_y   = (Y + 1) * 10000 + 101

    base_y = (
        T_trans
        .filter((F.col("VisitDateID") >= start_y) & (F.col("VisitDateID") < end_y))
        .select(
            "cde_id", "CineClubFLAG", "SCENEFLAG",
            "VisitDateID", "LocationId", "TicketFLAG", "ConcessionFLAG", "Net_Revenue"
        )
    )

    # Members who existed in that year
    existed_ids = base_y.select("cde_id").distinct()

    # Members who were CC in that year
    cc_ids = (
        base_y
        .filter(F.col("CineClubFLAG") == "Yes")
        .filter(F.col("SCENEFLAG") == "Yes")
        .select("cde_id")
        .distinct()
    )

    # Target cohort
    target_ids = (
        pure_cc_2025
        .join(existed_ids, "cde_id", "inner")
        .join(cc_ids, "cde_id", "left_anti")
    )

    # Previous year KPIs
    prev_metrics = (
        base_y
        .join(target_ids, "cde_id", "inner")
        .agg(
            F.countDistinct("cde_id").alias("Member_Count"),
            F.countDistinct(F.struct("cde_id", "VisitDateID", "LocationId")).alias("Total_Visits_Prev_Year"),
            F.sum(F.when(F.col("TicketFLAG") == "Yes", 1).otherwise(0)).alias("Total_Tickets_Prev_Year"),
            F.sum(
                F.when(F.col("TicketFLAG") == "Yes", F.coalesce(F.col("Net_Revenue"), F.lit(0.0)))
            ).alias("Ticket_Revenue_Prev_Year"),
            F.sum(
                F.when(F.col("ConcessionFLAG") == "Yes", F.coalesce(F.col("Net_Revenue"), F.lit(0.0)))
            ).alias("Concession_Revenue_Prev_Year")
        )
    )

    # 2025 KPIs for same cohort
    metrics_2025 = (
        base_2025
        .join(target_ids, "cde_id", "inner")
        .agg(
            F.countDistinct(F.struct("cde_id", "VisitDateID", "LocationId")).alias("Total_Visits_2025"),
            F.sum(F.when(F.col("TicketFLAG") == "Yes", 1).otherwise(0)).alias("Total_Tickets_2025"),
            F.sum(
                F.when(F.col("TicketFLAG") == "Yes", F.coalesce(F.col("Net_Revenue"), F.lit(0.0)))
            ).alias("Ticket_Revenue_2025"),
            F.sum(
                F.when(F.col("ConcessionFLAG") == "Yes", F.coalesce(F.col("Net_Revenue"), F.lit(0.0)))
            ).alias("Concession_Revenue_2025")
        )
    )

    out = (
        prev_metrics
        .crossJoin(metrics_2025)
        .withColumn("Prev_Year", F.lit(Y))
        .select(
            "Prev_Year",
            "Member_Count",
            "Total_Visits_Prev_Year",
            "Total_Visits_2025",
            "Total_Tickets_Prev_Year",
            "Total_Tickets_2025",
            "Ticket_Revenue_Prev_Year",
            "Ticket_Revenue_2025",
            "Concession_Revenue_Prev_Year",
            "Concession_Revenue_2025"
        )
    )

    outs.append(out)

df_result = reduce(lambda a, b: a.unionByName(b), outs)

display(df_result.orderBy("Prev_Year"))

In [0]:
from pyspark.sql import functions as F
from functools import reduce

base_year = 2025
compare_years = [2018,2019, 2020]

start_2025 = base_year * 10000 + 101
end_2025   = (base_year + 1) * 10000 + 101

# ----------------------------
# Base 2025 transactions
# ----------------------------
base_2025 = (
    T_trans
    .filter((F.col("VisitDateID") >= start_2025) & (F.col("VisitDateID") < end_2025))
    .select(
        "cde_id", "CineClubFLAG", "VisitDateID", "LocationId",
        "TicketFLAG", "ConcessionFLAG", "Net_Revenue", "Quantity"
    )
)

# ----------------------------
# Pure CineClub members in 2025
# ----------------------------
non_cc_ids_2025 = (
    base_2025
    .filter((F.col("CineClubFLAG") != "Yes") | F.col("CineClubFLAG").isNull())
    .select("cde_id")
    .distinct()
)

pure_cc_2025 = (
    base_2025
    .filter(F.col("CineClubFLAG") == "Yes")
    .select("cde_id")
    .distinct()
    .join(non_cc_ids_2025, "cde_id", "left_anti")
)

outs = []

for Y in compare_years:

    start_y = Y * 10000 + 101
    end_y   = (Y + 1) * 10000 + 101

    base_y = (
        T_trans
        .filter((F.col("VisitDateID") >= start_y) & (F.col("VisitDateID") < end_y))
        .select(
            "cde_id", "CineClubFLAG", "VisitDateID", "LocationId",
            "TicketFLAG", "ConcessionFLAG", "Net_Revenue", "Quantity"
        )
    )

    # Members who existed in compared year
    existed_ids = base_y.select("cde_id").distinct()

    # Members who were CC at all in compared year
    cc_ids_y = (
        base_y
        .filter(F.col("CineClubFLAG") == "Yes")
        .select("cde_id")
        .distinct()
    )

    # Cohort:
    # pure CC in 2025
    # existed in Y
    # but were NOT CC at all in Y
    target_ids = (
        pure_cc_2025
        .join(existed_ids, "cde_id", "inner")
        .join(cc_ids_y, "cde_id", "left_anti")
    )

    # ----------------------------
    # KPI in compared year
    # ----------------------------
    kpi_y = (
        base_y
        .join(target_ids, "cde_id", "inner")
        .agg(
            F.countDistinct("cde_id").alias("Members"),
            F.countDistinct(F.struct("cde_id", "VisitDateID", "LocationId")).alias(f"Total_Visits_{Y}"),
             F.sum(
                F.when(
                    F.col("TicketFLAG") == "Yes",
                    F.coalesce(F.col("Quantity"), F.lit(0.0))
                ).otherwise(F.lit(0.0))
            ).alias(f"Total_Tickets_{Y}"),
            F.sum(
                F.when(
                    F.col("TicketFLAG") == "Yes",
                    F.coalesce(F.col("Net_Revenue"), F.lit(0.0))
                ).otherwise(F.lit(0.0))
            ).alias(f"Net_Ticket_Revenue_{Y}"),
            F.sum(
                F.when(
                    F.col("ConcessionFLAG") == "Yes",
                    F.coalesce(F.col("Net_Revenue"), F.lit(0.0))
                ).otherwise(F.lit(0.0))
            ).alias(f"Net_Concession_Revenue_{Y}")
        )
    )

    # ----------------------------
    # KPI in 2025
    # ----------------------------
    kpi_2025 = (
        base_2025
        .join(target_ids, "cde_id", "inner")
        .agg(
            F.countDistinct(F.struct("cde_id", "VisitDateID", "LocationId")).alias("Total_Visits_2025"),
            F.sum(
                F.when(
                    F.col("TicketFLAG") == "Yes",
                    F.coalesce(F.col("Quantity"), F.lit(0.0))
                ).otherwise(F.lit(0.0))
            ).alias("Total_Tickets_2025"),
            F.sum(
                F.when(
                    F.col("TicketFLAG") == "Yes",
                    F.coalesce(F.col("Net_Revenue"), F.lit(0.0))
                ).otherwise(F.lit(0.0))
            ).alias("Net_Ticket_Revenue_2025"),
            F.sum(
                F.when(
                    F.col("ConcessionFLAG") == "Yes",
                    F.coalesce(F.col("Net_Revenue"), F.lit(0.0))
                ).otherwise(F.lit(0.0))
            ).alias("Net_Concession_Revenue_2025")
        )
    )

    out = (
        kpi_y
        .crossJoin(kpi_2025)
        .withColumn("Compared_Year", F.lit(Y))
        .withColumn("Base_Year", F.lit(2025))
        .select(
            "Compared_Year",
            "Base_Year",
            "Members",
            f"Total_Visits_{Y}",
            "Total_Visits_2025",
            f"Total_Tickets_{Y}",
            "Total_Tickets_2025",
            f"Net_Ticket_Revenue_{Y}",
            "Net_Ticket_Revenue_2025",
            f"Net_Concession_Revenue_{Y}",
            "Net_Concession_Revenue_2025"
        )
    )

    outs.append(out)

df_result_compare = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), outs)

In [0]:
display(df_result_compare.orderBy("Compared_Year"))

## Redemption Analysis

In [0]:
df_cc_redemptions = (
    T_trans
    .filter(F.col("VisitDateId") < "20260301")
    .filter(F.col("TicketFLAG") == "Yes")
    .filter(F.col("CineClubFLAG") == "Yes")
    .filter(F.col("CineClub_Discount_Type_ID") == "6")   # Free CC ticket
    .withColumn(
        "Redemption_Date",
        F.to_date(F.col("VisitDateId").cast("string"), "yyyyMMdd")
    )
    .groupBy("CDE_ID", "Redemption_Date", "LocationId")
    .agg(
        F.sum(F.col("Quantity")).alias("Redemption_Count")
    )
)


In [0]:
df_cc_redemptions.display()
df_cc_redemptions.cache()

In [0]:
df_subscriber_bridge = (
    T_subscriber
    .select("CDE_ID", "cc_user_profile_id")
)

In [0]:
df_subscriptions = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/data_exploration_hub/marketing_hub/offline_data/Subscriptions - FEB 2026 (Incrementality Analysis).csv")
)

date_format = "M/d/yyyy H:mm"

df_subscriptions = (
    df_subscriptions
    .withColumn("`subscriptions.started_at`", 
        F.to_timestamp(F.col("`subscriptions.started_at`"), date_format))
    .withColumn("`subscriptions.cancelled_at`", 
        F.to_timestamp(F.col("`subscriptions.cancelled_at`"), date_format))
    .withColumn("`subscriptions.current_term_end`", 
        F.to_timestamp(F.col("`subscriptions.current_term_end`"), date_format))
)

In [0]:
date_format = "M/d/yyyy H:mm"

df_sub_profile = (
    df_subscriptions
    .select(
        F.col("`customers.id`").alias("cc_user_profile_id"),
        F.to_date(F.col("`subscriptions.started_at`"), date_format).alias("subscription_start_date"),
        F.to_date(F.col("`subscriptions.cancelled_at`"), date_format).alias("cancelled_at"),
        F.col("`subscriptions.status`").alias("subscription_status"),
        F.to_date(F.col("`subscriptions.current_term_end`"), date_format).alias("current_term_end")
    )
    .withColumn(
        "is_cancelled",
        F.when(F.col("subscription_status") == "Cancelled", True).otherwise(False)
    )
    .filter(F.col("`customers.id`") != "AzqBunUAsPVB41nKT")

)

In [0]:
df_member_profile = (
    df_subscriber_bridge
    .join(df_sub_profile, on="cc_user_profile_id", how="left")
)

In [0]:
df_analysis = (
    df_cc_redemptions
    .join(df_member_profile, on="CDE_ID", how="left")
    .withColumn(
        "days_to_redemption",
        F.datediff(F.col("Redemption_Date"), F.col("subscription_start_date"))
    )
    .withColumn(
        "months_to_redemption",
        (F.col("days_to_redemption") / 30.44).cast("int")
    )
    # Tickets earned = 1 per month since subscription start (uniform across plans)
    .withColumn(
        "tickets_earned_at_redemption",
        (F.datediff(F.col("Redemption_Date"), F.col("subscription_start_date")) / 30.44).cast("int")
    )
    # Flag redemptions that occurred after cancellation
    .withColumn(
        "redeemed_post_cancellation",
        F.when(
            F.col("is_cancelled") & 
            (F.col("Redemption_Date") > F.col("cancelled_at")),
            True
        ).otherwise(False)
    )
    # Days between cancellation and redemption (null if not cancelled or redeemed before)
    .withColumn(
        "days_post_cancellation",
        F.when(
            F.col("redeemed_post_cancellation"),
            F.datediff(F.col("Redemption_Date"), F.col("cancelled_at"))
        ).otherwise(None)
    )
    # Bucket by months_to_redemption
    .withColumn(
        "redemption_bucket",
        F.when(F.col("months_to_redemption") <= 1,  "Month 1")
         .when(F.col("months_to_redemption") <= 2,  "Month 2")
         .when(F.col("months_to_redemption") <= 3,  "Month 3")
         .when(F.col("months_to_redemption") <= 6,  "Months 4–6")
         .when(F.col("months_to_redemption") <= 12, "Months 7–12")
         .otherwise("12+ Months")
    )
)

# Step 5: Summary — member count per redemption bucket
df_summary = (
    df_analysis
    .groupBy("redemption_bucket", "is_cancelled")
    .agg(
        F.countDistinct("CDE_ID").alias("unique_members"),
        F.sum("Redemption_Count").alias("total_redemptions"),
        F.avg("days_to_redemption").alias("avg_days_to_redemption"),
        F.avg("days_post_cancellation").alias("avg_days_post_cancellation")
    )
    .orderBy("redemption_bucket")
)

In [0]:
df_summary.display()

In [0]:
# Add sort key for logical bucket ordering
df_summary_clean = (
    df_summary
    .withColumn(
        "bucket_order",
        F.when(F.col("redemption_bucket") == "Month 1", 1)
         .when(F.col("redemption_bucket") == "Month 2", 2)
         .when(F.col("redemption_bucket") == "Month 3", 3)
         .when(F.col("redemption_bucket") == "Months 4–6", 4)
         .when(F.col("redemption_bucket") == "Months 7–12", 5)
         .when(F.col("redemption_bucket") == "12+ Months", 6)
         .otherwise(99)
    )
    .withColumn(
        "member_type",
        F.when(F.col("is_cancelled") == True, "cancelled")
         .when(F.col("is_cancelled") == False, "active")
         .otherwise("unknown")
    )
    .groupBy("redemption_bucket", "bucket_order")
    .agg(
        F.sum(F.when(F.col("member_type") == "active",   F.col("unique_members"))).alias("Active Members"),
        F.sum(F.when(F.col("member_type") == "cancelled", F.col("unique_members"))).alias("Cancelled Members"),
        F.sum(F.when(F.col("member_type") == "unknown",  F.col("unique_members"))).alias("Unknown Members"),
        F.round(F.avg(F.when(F.col("member_type") == "active",    F.col("avg_days_to_redemption"))), 0).alias("Avg Days to Redeem (Active)"),
        F.round(F.avg(F.when(F.col("member_type") == "cancelled", F.col("avg_days_to_redemption"))), 0).alias("Avg Days to Redeem (Cancelled)"),
        F.round(F.avg(F.when(F.col("member_type") == "cancelled", F.col("avg_days_post_cancellation"))), 0).alias("Avg Days Post Cancellation"),
    )
    .orderBy("bucket_order")
    .drop("bucket_order")
)

df_summary_clean.display()

In [0]:
df_analysis = (
    df_cc_redemptions
    .join(df_member_profile, on="CDE_ID", how="left")
    .filter(F.col("Redemption_Date") >= F.col("subscription_start_date"))  # Only redemptions after start date
    .withColumn(
        "days_to_redemption",
        F.datediff(F.col("Redemption_Date"), F.col("subscription_start_date"))
    )
    .withColumn(
        "months_to_redemption",
        (F.col("days_to_redemption") / 30.44).cast("int")
    )
    # Tickets earned = 1 per month since subscription start (uniform across plans)
    .withColumn(
        "tickets_earned_at_redemption",
        (F.datediff(F.col("Redemption_Date"), F.col("subscription_start_date")) / 30.44).cast("int")
    )
    # Flag redemptions that occurred after cancellation
    .withColumn(
        "redeemed_post_cancellation",
        F.when(
            F.col("is_cancelled") & 
            (F.col("Redemption_Date") > F.col("cancelled_at")),
            True
        ).otherwise(False)
    )
    # Days between cancellation and redemption (null if not cancelled or redeemed before)
    .withColumn(
        "days_post_cancellation",
        F.when(
            F.col("redeemed_post_cancellation"),
            F.datediff(F.col("Redemption_Date"), F.col("cancelled_at"))
        ).otherwise(None)
    )
    # Bucket by months_to_redemption
    .withColumn(
        "redemption_bucket",
        F.when(F.col("months_to_redemption") <= 1,  "Month 1")
         .when(F.col("months_to_redemption") <= 2,  "Month 2")
         .when(F.col("months_to_redemption") <= 3,  "Month 3")
         .when(F.col("months_to_redemption") <= 6,  "Months 4–6")
         .when(F.col("months_to_redemption") <= 12, "Months 7–12")
         .otherwise("12+ Months")
    )
)

from pyspark.sql import Window

# Keep only the first redemption per member
window_first_redemption = Window.partitionBy("CDE_ID").orderBy("Redemption_Date")

df_summary = (
    df_analysis
    .withColumn("redemption_rank", F.rank().over(window_first_redemption))
    .filter(F.col("redemption_rank") == 1)
    .drop("redemption_rank")
    .groupBy("redemption_bucket", "is_cancelled")
    .agg(
        F.countDistinct("CDE_ID").alias("unique_members"),
        F.sum("Redemption_Count").alias("total_redemptions"),
        F.avg("days_to_redemption").alias("avg_days_to_redemption"),
        F.avg("days_post_cancellation").alias("avg_days_post_cancellation")
    )
    .orderBy("redemption_bucket")
)

df_summary_clean = (
    df_summary
    .withColumn(
        "bucket_order",
        F.when(F.col("redemption_bucket") == "Month 1", 1)
         .when(F.col("redemption_bucket") == "Month 2", 2)
         .when(F.col("redemption_bucket") == "Month 3", 3)
         .when(F.col("redemption_bucket") == "Months 4–6", 4)
         .when(F.col("redemption_bucket") == "Months 7–12", 5)
         .when(F.col("redemption_bucket") == "12+ Months", 6)
         .otherwise(99)
    )
    .withColumn(
        "member_type",
        F.when(F.col("is_cancelled") == True, "cancelled")
         .when(F.col("is_cancelled") == False, "active")
         .otherwise("unknown")
    )
    .groupBy("redemption_bucket", "bucket_order")
    .agg(
        F.sum(F.when(F.col("member_type") == "active",   F.col("unique_members"))).alias("Active Members"),
        F.sum(F.when(F.col("member_type") == "cancelled", F.col("unique_members"))).alias("Cancelled Members"),
        F.sum(F.when(F.col("member_type") == "unknown",  F.col("unique_members"))).alias("Unknown Members"),
        F.round(F.avg(F.when(F.col("member_type") == "active",    F.col("avg_days_to_redemption"))), 0).alias("Avg Days to Redeem (Active)"),
        F.round(F.avg(F.when(F.col("member_type") == "cancelled", F.col("avg_days_to_redemption"))), 0).alias("Avg Days to Redeem (Cancelled)"),
        F.round(F.avg(F.when(F.col("member_type") == "cancelled", F.col("avg_days_post_cancellation"))), 0).alias("Avg Days Post Cancellation"),
    )
    .orderBy("bucket_order")
    .drop("bucket_order")
)


## CINEMARK Model

In [0]:
base_year  = 2024
start_date = base_year * 10000 + 101    # 20240101
end_date   = base_year * 10000 + 1231   # 20241231

# ----------------------------
# Non-CC visitors in 2024
# Single scan, no intermediate views
# ----------------------------
non_cc_visitors_2024 = spark.sql(f"""
    WITH yearly_transactions AS (
        SELECT
            cde_id, CineClubFLAG, VisitDateID, LocationId,
            TicketFLAG, ConcessionFLAG, Net_Revenue, Quantity, SCENEFLAG
        FROM cpx_dataanalytics_gold.customer360.c360_f_transaction
        WHERE VisitDateID BETWEEN {start_date} AND {end_date}
        AND SCENEFLAG = 'Yes'
    ),
    cc_members AS (
        SELECT DISTINCT cde_id
        FROM yearly_transactions
        WHERE CineClubFLAG = 'Yes'
    )
    SELECT DISTINCT t.cde_id
    FROM yearly_transactions t
    LEFT ANTI JOIN cc_members cc
        ON t.cde_id = cc.cde_id
""")

In [0]:
display(non_cc_visitors_2024.count())

In [0]:
start_date_2025 = 20250101
end_date_2025   = 20251231

non_cc_visitors_2024.createOrReplaceTempView("non_cc_visitors_2024")

# ----------------------------
# 2025 behaviour of 2024 Non-CC population
# Segmented by Month and CC vs Non-CC
# CC status evaluated per person per month
# ----------------------------
visitor_analysis_2025 = spark.sql(f"""
    WITH transactions_2025 AS (
        SELECT
            t.cde_id,
            t.VisitDateID,
            t.LocationId,
            t.TicketFLAG,
            t.ConcessionFLAG,
            t.Net_Revenue,
            t.Quantity,
            CAST(SUBSTR(CAST(t.VisitDateID AS STRING), 1, 6) AS INT) AS visit_month,
            CASE
                WHEN t.CineClubFLAG = 'Yes' THEN 'CC Member'
                ELSE 'Non-CC Member'
            END AS member_type
        FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
        INNER JOIN non_cc_visitors_2024 nc
            ON t.cde_id = nc.cde_id
        WHERE t.VisitDateID BETWEEN {start_date_2025} AND {end_date_2025}
    ),
    cc_flag_per_member_month AS (
        SELECT DISTINCT
            cde_id,
            visit_month,
            member_type
        FROM transactions_2025
        WHERE member_type = 'CC Member'
    )
    SELECT
        t.visit_month,
        COALESCE(cf.member_type, 'Non-CC Member')                                  AS member_type,

        COUNT(DISTINCT t.cde_id)                                                    as unique_members,

        -- Visits: distinct combo of cde_id, VisitDateID, LocationId
        COUNT(DISTINCT CONCAT(t.cde_id, '-', t.VisitDateID, '-', t.LocationId))   AS visits,

        -- Ticket Count
        SUM(CASE WHEN t.TicketFLAG = 'Yes' THEN t.Quantity ELSE 0 END)            AS ticket_count,

        -- Ticket Revenue
        SUM(CASE WHEN t.TicketFLAG = 'Yes' THEN t.Net_Revenue ELSE 0 END)         AS ticket_revenue,

        -- Concession Revenue
        SUM(CASE WHEN t.ConcessionFLAG = 'Yes' THEN t.Net_Revenue ELSE 0 END)     AS concession_revenue

    FROM transactions_2025 t
    LEFT JOIN cc_flag_per_member_month cf
        ON  t.cde_id     = cf.cde_id
        AND t.visit_month = cf.visit_month
    GROUP BY
        t.visit_month,
        COALESCE(cf.member_type, 'Non-CC Member')
    ORDER BY
        t.visit_month,
        member_type
""")

In [0]:
visitor_analysis_2025.display()

In [0]:
start_date_2025 = 20250101
end_date_2025   = 20251231

non_cc_visitors_2024.createOrReplaceTempView("non_cc_visitors_2024")

# ----------------------------
# 2025 behaviour of 2024 Non-CC population
# Once CC in any month → CC for all subsequent months
# ----------------------------
visitor_analysis_2025 = spark.sql(f"""
    WITH transactions_2025 AS (
        SELECT
            t.cde_id,
            t.VisitDateID,
            t.LocationId,
            t.TicketFLAG,
            t.ConcessionFLAG,
            t.Net_Revenue,
            t.Quantity,
            CAST(SUBSTR(CAST(t.VisitDateID AS STRING), 1, 6) AS INT) AS visit_month,
            t.CineClubFLAG
        FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
        INNER JOIN non_cc_visitors_2024 nc
            ON t.cde_id = nc.cde_id
        WHERE t.VisitDateID BETWEEN {start_date_2025} AND {end_date_2025}
    ),

    -- First month each person became a CC Member
    first_cc_month AS (
        SELECT
            cde_id,
            MIN(visit_month) AS cc_start_month
        FROM transactions_2025
        WHERE CineClubFLAG = 'Yes'
        GROUP BY cde_id
    ),

    -- Flag each person per month:
    -- CC Member if visit_month >= their first CC month
    -- Non-CC Member otherwise (or if never CC)
    member_month_classified AS (
        SELECT
            t.cde_id,
            t.visit_month,
            t.VisitDateID,
            t.LocationId,
            t.TicketFLAG,
            t.ConcessionFLAG,
            t.Net_Revenue,
            t.Quantity,
            CASE
                WHEN t.visit_month >= fc.cc_start_month THEN 'CC Member'
                ELSE 'Non-CC Member'
            END AS member_type
        FROM transactions_2025 t
        LEFT JOIN first_cc_month fc
            ON t.cde_id = fc.cde_id
    )

    SELECT
        visit_month,
        member_type,

        -- Member Count
        COUNT(DISTINCT cde_id)                                                   AS member_count,

        -- Visits: distinct combo of cde_id, VisitDateID, LocationId
        COUNT(DISTINCT CONCAT(cde_id, '-', VisitDateID, '-', LocationId))        AS visits,

        -- Ticket Count
        SUM(CASE WHEN TicketFLAG = 'Yes' THEN Quantity ELSE 0 END)              AS ticket_count,

        -- Ticket Revenue
        SUM(CASE WHEN TicketFLAG = 'Yes' THEN Net_Revenue ELSE 0 END)           AS ticket_revenue,

        -- Concession Revenue
        SUM(CASE WHEN ConcessionFLAG = 'Yes' THEN Net_Revenue ELSE 0 END)       AS concession_revenue

    FROM member_month_classified
    GROUP BY
        visit_month,
        member_type
    ORDER BY
        visit_month,
        member_type
""")

In [0]:
visitor_analysis_2025.display()
